# Combined Model Evaluation
This notebook loads both trained models (audio + linguistic) 
and evaluates them on the same test subjects to compare performance.

In [1]:
import sys
print(sys.executable)

c:\Users\MANSI\AppData\Local\Programs\Python\Python311\python.exe


In [21]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [22]:
# Assume audio_df is your audio dataframe
# Exclude the last column 'label' for features
X_audio = audio_df.iloc[:, :-1].values

# Labels
y_audio = audio_df['label'].values

print("Audio features shape:", X_audio.shape)
print("Labels shape:", y_audio.shape)

Audio features shape: (864, 26)
Labels shape: (864,)


In [23]:
# Make sure text_proba has same number of rows as audio
# If text_proba has fewer rows, take first N rows to match
X_text = text_proba[:len(X_audio)]

print("Text probabilities shape:", X_text.shape)

Text probabilities shape: (399, 2)


In [24]:
# Align audio to match text row count
X_audio_aligned = X_audio[:len(X_text)]
y_audio_aligned = y_audio[:len(X_text)]

# Now concatenate
X_combined = np.hstack([X_audio_aligned, X_text])
print("Combined features shape:", X_combined.shape)

Combined features shape: (399, 28)


In [25]:
# Use aligned data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_audio_aligned, test_size=0.2, random_state=42, stratify=y_audio_aligned
)

# Train classifier
clf = RandomForestClassifier(class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.82      0.88        22
           1       0.93      0.98      0.96        58

    accuracy                           0.94        80
   macro avg       0.94      0.90      0.92        80
weighted avg       0.94      0.94      0.94        80

